# บทที่ 4 · วันที่และเวลา

งานข้อมูลแทบทุกงานเกี่ยวกับเวลา: "ยอดขายของเมื่อวาน", "ย้อนหลัง 7 วัน", "ทุกวันตอนตี 1"

บทนี้: **date · datetime · timedelta · แปลงข้อความ ↔ วันที่ · ช่วงเวลาของ 1 วัน · วนทีละวัน · timezone**

## 4.1 `date`, `datetime`, `timedelta`

In [ ]:
from datetime import date, datetime, timedelta

payday = date(2026, 9, 30)                          # วันที่อย่างเดียว
meeting = datetime(2026, 9, 25, 14, 30)             # วันที่ + เวลา
print(payday, "|", meeting)
print(meeting.year, meeting.month, meeting.day, meeting.hour, meeting.minute)
print(meeting.date(), meeting.time())

print("ตอนนี้:", datetime.now())
print("วันนี้:", date.today())

In [ ]:
# timedelta = "ระยะเวลา" เอาไปบวก/ลบกับวันที่ได้
today = date(2026, 9, 25)
print("พรุ่งนี้:", today + timedelta(days=1))
print("เมื่อวาน:", today - timedelta(days=1))
print("อีก 2 สัปดาห์:", today + timedelta(weeks=2))
print("ประชุมเลิก:", meeting + timedelta(hours=1, minutes=30))

In [ ]:
# วันที่ลบวันที่ = ระยะห่าง
days_to_payday = payday - today
print(days_to_payday, "|", days_to_payday.days, "วันกว่าเงินเดือนออก")
print(today < payday)                               # เปรียบเทียบได้

In [ ]:
# อายุ (แบบง่าย)
birthday = date(2001, 5, 14)
age_days = (today - birthday).days
print(f"เกิดมาแล้ว {age_days:,} วัน ≈ {age_days // 365} ปี")

## 4.2 แปลง วันที่ ↔ ข้อความ

ข้อมูลจากไฟล์/เว็บมาเป็น **ข้อความ** เสมอ ต้องแปลงเป็นวันที่ก่อนถึงจะบวกลบได้

In [ ]:
# วันที่ -> ข้อความ
print(today.isoformat())                    # '2026-09-25'  รูปแบบมาตรฐานสากล (ปี-เดือน-วัน)
print(meeting.isoformat())                  # '2026-09-25T14:30:00'
print(meeting.strftime("%d/%m/%Y %H:%M"))   # แบบที่คนไทยคุ้น
print(today.strftime("%Y%m%d"))             # ไว้ตั้งชื่อไฟล์ เช่น photo_20260925.jpg

# ข้อความ -> วันที่
print(date.fromisoformat("2026-10-01"))
print(datetime.fromisoformat("2026-09-25 14:30:00"))
print(datetime.strptime("25/09/2026", "%d/%m/%Y"))       # รูปแบบอื่น: บอกรูปแบบเอง

| รหัส | ความหมาย | ตัวอย่าง |
|---|---|---|
| `%Y` `%m` `%d` | ปี เดือน วัน | 2026 09 25 |
| `%H` `%M` `%S` | ชั่วโมง นาที วินาที | 14 30 00 |

💡 เก็บ/ส่งวันที่เป็นรูปแบบ `YYYY-MM-DD` เสมอ เรียงตามตัวอักษรแล้วได้ลำดับเวลาถูกต้องด้วย

## 4.3 ช่วงเวลาของ "1 วัน" : `[เริ่ม, จบ)`

"ยอดขายของวันที่ 25" = ตั้งแต่ **25 เวลา 00:00** จนถึง **ก่อน** 26 เวลา 00:00
(รวมต้น ไม่รวมปลาย) ไม่พลาดรายการวินาทีสุดท้าย และไม่นับซ้ำกับวันถัดไป

In [ ]:
def day_range(day: str) -> tuple:
    start = datetime.fromisoformat(day)      # 2026-09-25 00:00:00
    end = start + timedelta(days=1)          # 2026-09-26 00:00:00
    return start, end


start, end = day_range("2026-09-25")
print(start, "->", end)

sales_times = [datetime(2026, 9, 24, 23, 59), datetime(2026, 9, 25, 0, 0), datetime(2026, 9, 25, 23, 59, 59),
               datetime(2026, 9, 26, 0, 0)]
for t in sales_times:
    print(t, "นับเป็นของวันที่ 25?", start <= t < end)

## 4.4 วนทีละวัน

In [ ]:
def list_days(first_day: str, last_day: str) -> list[str]:
    """ทุกวันตั้งแต่ first_day ถึง last_day (รวมทั้งสองวัน)"""
    current = date.fromisoformat(first_day)
    last = date.fromisoformat(last_day)
    days = []
    while current <= last:
        days.append(current.isoformat())
        current += timedelta(days=1)
    return days


trip_days = list_days("2026-09-28", "2026-10-02")
print(trip_days)
for d in trip_days:
    print(d, date.fromisoformat(d).strftime("%A"))      # %A = ชื่อวันในสัปดาห์

## 4.5 Timezone ⚠️

- เวลาที่ **ไม่บอก** timezone เรียกว่า naive: `datetime(2026, 9, 25, 9, 0)` ไม่รู้ว่า 9 โมงที่ไหน
- เวลาที่ **บอก** timezone : 9 โมงที่กรุงเทพ = 2 โมงเช้าที่ลอนดอน (UTC)
- ระบบคอมพิวเตอร์/เซิร์ฟเวอร์จำนวนมากเก็บเวลาเป็น **UTC** (ไทย = UTC+7)

In [ ]:
from zoneinfo import ZoneInfo

BANGKOK = ZoneInfo("Asia/Bangkok")
UTC = ZoneInfo("UTC")
TOKYO = ZoneInfo("Asia/Tokyo")

match_time = datetime(2026, 9, 26, 1, 0, tzinfo=BANGKOK)     # ฟุตบอลเตะตี 1 วันที่ 26 เวลาไทย
print("กรุงเทพ:", match_time)
print("UTC    :", match_time.astimezone(UTC))
print("โตเกียว:", match_time.astimezone(TOKYO))
print("วันที่ตาม UTC:", match_time.astimezone(UTC).date(), " วันที่ตามไทย:", match_time.date())   # คนละวัน!

💡 ถ้าระบบให้เวลาเป็น UTC มา แต่ข้อมูลของเราเป็นเวลาไทย ให้ **แปลงเป็นเวลาไทยก่อน** แล้วค่อยคิดว่าเป็นวันไหน
และถ้าจะบวก/ลบ "วัน" ให้แปลงเป็น `date` ก่อน ปลอดภัยที่สุด

In [ ]:
server_time = datetime(2026, 9, 25, 18, 0, tzinfo=UTC)          # เซิร์ฟเวอร์บอกเวลาเป็น UTC
thai_day = server_time.astimezone(BANGKOK).date()               # = 26 ก.ย. ตามเวลาไทย
yesterday = thai_day - timedelta(days=1)
print("วันนี้ (ไทย):", thai_day, " เมื่อวาน:", yesterday)

## ✅ สรุปบทที่ 4

| เรื่อง | ตัวอย่าง |
|---|---|
| สร้าง | `date(2026, 9, 25)`, `datetime(2026, 9, 25, 14, 30)`, `date.today()` |
| บวกลบ | `today + timedelta(days=1)`, `payday - today` |
| วันที่ → ข้อความ | `.isoformat()`, `.strftime("%d/%m/%Y")` |
| ข้อความ → วันที่ | `date.fromisoformat("2026-09-25")`, `datetime.strptime(text, "%d/%m/%Y")` |
| 1 วัน | `start <= t < start + timedelta(days=1)` |
| timezone | `datetime(..., tzinfo=ZoneInfo("Asia/Bangkok"))`, `.astimezone(...)` |

🧪 **แบบฝึกท้ายบท:** เขียน function `last_7_days(end_day: str) -> list[str]` คืนวันที่ 7 วันย้อนหลังที่จบที่ `end_day`
(รวม end_day, เรียงจากเก่าไปใหม่) เช่น `last_7_days("2026-10-01")` → `["2026-09-25", ..., "2026-10-01"]`

In [ ]:
# เขียนต่อตรงนี้